In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a GPU program that computes the Rotary Positional Embedding (RoPE) for a batch of query vectors.
    RoPE is a method for encoding positional information in transformer models by rotating the query and key vectors using precomputed cosine and sine components.
</p>
<p>
    Mathematically, given a query vector $x$ and corresponding cosine and sine vectors, the operation is defined as:
    $$
    \text{RoPE}(x) = x \odot \cos + \text{rotate\_half}(x) \odot \sin
    $$
</p>
<p>
    Where $\odot$ denotes element-wise multiplication. The $\text{rotate\_half}(x)$ operation swaps the first and second halves of the vector and negates the first half. For a vector of dimension $d$:
    $$
    \text{rotate\_half}([x_1, \dots, x_{d/2}, x_{d/2+1}, \dots, x_d]) = [-x_{d/2+1}, \dots, -x_d, x_1, \dots, x_{d/2}]
    $$
</p>
<h2>Implementation Requirements</h2>
<ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The input tensors <code>Q</code>, <code>cos</code>, and <code>sin</code> have shape <code>(M, D)</code>, where <code>M</code> is the number of tokens and <code>D</code> is the head dimension</li>
    <li><code>D</code> (head dimension) is guaranteed to be an even number</li>
    <li>The final result must be stored in the output variable with the same shape <code>(M, D)</code></li>
</ul>
<h2>Example 1:</h2>
<pre>Input:  Q   = [[1.0, 2.0, 3.0, 4.0],
               [1.0, 1.0, 1.0, 1.0]]
        Cos = [[1.0, 1.0, 1.0, 1.0],
               [0.0, 0.0, 0.0, 0.0]]
        Sin = [[0.0, 0.0, 0.0, 0.0],
               [1.0, 1.0, 1.0, 1.0]]
Output: result = [[1.0, 2.0, 3.0, 4.0],
                  [-1.0, -1.0, 1.0, 1.0]]
        (Row 0 is identity via Cos; Row 1 is rotated via Sin)</pre>
<h2>Constraints</h2>
<ul>
    <li><code>Q</code>, <code>cos</code>, and <code>sin</code> have identical dimensions</li>
    <li><code>D</code> % 2 == 0</li>
    <li>1 ≤ <code>M</code>, <code>D</code> ≤ 10,000</li>

  <li>Performance is measured with <code>D</code> = 128, <code>M</code> = 1,048,576</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, cos, sin, output are device pointers
extern "C" void solve(float* Q, float* cos, float* sin, float* output, int M, int D) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# Q, cos, sin, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    cos: cute.Tensor,
    sin: cute.Tensor,
    output: cute.Tensor,
    M: cute.Int32,
    D: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# Q, cos, sin are tensors on the GPU
@jax.jit
def solve(Q: jax.Array, cos: jax.Array, sin: jax.Array, M: int, D: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# Q, cos, sin, output are device pointers
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    cos: UnsafePointer[Float32, MutExternalOrigin],
    sin: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    D: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# Q, cos, sin, output are tensors on the GPU
def solve(
    Q: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, output: torch.Tensor, M: int, D: int
):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# Q, cos, sin, output are tensors on the GPU
def solve(
    Q: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, output: torch.Tensor, M: int, D: int
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/61_rope_embedding/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
